# SBERT Sentence Analysis

## 1. Preparations

### 1.1 Read Data
This new data is obtained after some further anonymization, and by putting previously incorrectly split sentences together, and removed duplicates

In [1]:
import pandas as pd
# read the sentence data 
df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/sentence_data_for_analysis.xlsx", index_col=0)
sentences = df["sentence"].to_list()
# check the df head
df.head()

,message_id,sentence,clean_sentence,sentence_id,translated_sentence
0,1,"Geachte ibd groep, Is mijn uitslag al binnen ...","ibd groep, Is mijn uitslag al binnen van de b...",1,"Dear Ibd group, has my results come back from ..."
1,3,Vorige week is door [ZIEKENHUIS] [LOCATIE] mij...,Vorige week is door mijn ontlasting onderzoc...,2,Last week my stool was examined by [SIGHSHOUSE...
2,3,Graag zou ik de uitkomst hiervan vernemen.,Graag zou ik de uitkomst hiervan vernemen.,3,I would like to hear the outcome of this.
3,4,bloed in de ontlasting wordt steeds meer en st...,bloed in de ontlasting wordt steeds meer en st...,4,blood in the stool is becoming more and more f...
4,4,Ligt dit aan de medicatie?,Ligt dit aan de medicatie?,5,Is this because of the medication?


In [2]:
# check input size
len(sentences)

41119

### 1.2. Import the list of stopwords

In [2]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/persistent/mijnidbcoachnlp/data/analysis_data/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

# Extend the stop words if needed
extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl", "weken", "week", "dagen", "dag", "mg", "coach", "mijnibdcoach", "dr", "uur", "dgs"
]

dutch_stopwords.extend(extra_list)

### 1.4. Embed the lists of sentences

#### 1.4.1. sentence-transformers/distiluse-base-multilingual-cased-v1

In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm.autonotebook import tqdm  
import numpy as np
import sentence_transformers.util
sentence_transformers.util.tqdm = tqdm
# Now use your model
#embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")
# Generate and save embeddings
#embeddings = embedding_model.encode(sentences, show_progress_bar=True)

In [9]:
np.save('/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_sentence_placeholder.npy', embeddings)


NameError: name 'embeddings' is not defined

#### 1.4.2. sentence-transformers/distiluse-base-multilingual-cased-v2

In [10]:
# Now use your model
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v2")
# Generate and save embeddings
embeddings = embedding_model.encode(sentences, show_progress_bar=True)

Batches:   0%|          | 3/1285 [00:02<17:01,  1.25it/s]


KeyboardInterrupt: 

In [ ]:
np.save('/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_v2_sentence_placeholder.npy', embeddings)


#### 1.4.3. NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers

In [ ]:
# Now use your model
embedding_model = SentenceTransformer("NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers")
# Generate and save embeddings
embeddings = embedding_model.encode(sentences, show_progress_bar=True)

In [ ]:
np.save('/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_robbert2022_sentence_placeholder.npy', embeddings)

#### 1.4.4. sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

In [ ]:
# Now use your model
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# Generate and save embeddings
embeddings = embedding_model.encode(sentences, show_progress_bar=True)

In [ ]:
np.save('/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_miniL12v2_sentence_placeholder.npy', embeddings)

#### 1.4.5. sentence-transformers/paraphrase-multilingual-mpnet-base-v2

In [ ]:
# Now use your model
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
# Generate and save embeddings
embeddings = embedding_model.encode(sentences, show_progress_bar=True)

In [ ]:
np.save('/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_mpnet_v2_sentence_placeholder.npy', embeddings)

### 1.5. Function for stemming

In [3]:
from nltk.stem.snowball import SnowballStemmer
from tqdm import tqdm

# Create Dutch stemmer
stemmer = SnowballStemmer("dutch")

# Example
def stem_word(stemmer, words):
    stems = [stemmer.stem(word) for word in words]
    return stems

def stem_tokens(stemmer, tokenized_texts):
    list_stems = []
    for text in tqdm(tokenized_texts):
        stems = stem_word(stemmer, text)
        list_stems.append(stems)

    return list_stems


### 1.6. Tokenize the documents and lemmatize tokenized texts

In [8]:
# download nltk and spacy library
import nltk
from nltk.tokenize import word_tokenize
import string

# Download NLTK's tokenizer (only needed once)
nltk.download("punkt_tab")

# to calculate c_v, we need to import the original messages as the reference corpus
messages_df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/translated_clean_message_data.xlsx", index_col=0)
messages = messages_df["clean_message"].to_list()

#remove punctuations before tokenize
messages = [msg.translate(str.maketrans("", "", string.punctuation)) for msg in messages]

#tokenize the texts with nltk
tokenized_texts = [word_tokenize(message, language="dutch") for message in messages]

# save tokenized texts
import pickle

with open("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/tokenized_texts.pkl", "wb") as f:
    pickle.dump(tokenized_texts, f)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [14]:
# load tokenized texts and create Dictionary
import pickle
from gensim.corpora.dictionary import Dictionary

dictionary = Dictionary(tokenized_texts)
# stem the tokens
tokenized_texts_stem = stem_tokens(stemmer=stemmer, tokenized_texts=tokenized_texts)
stem_dictionary = Dictionary(tokenized_texts_stem)

with open("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/tokenized_texts_stem.pkl", "wb") as f:
    pickle.dump(tokenized_texts_stem, f)

100%|██████████| 32681/32681 [00:21<00:00, 1530.75it/s]


### 1.7 Load embeddings and  tokenized texts (with stem)

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Define each embedding model using SentenceTransformer
embedding_model_stv1 = SentenceTransformer("distiluse-base-multilingual-cased-v1")  # Adjust model name if needed
embedding_model_stv2 = SentenceTransformer("distiluse-base-multilingual-cased-v2")
embedding_model_stmini = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")  # Adjust this if needed
embedding_model_stmp = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")  # Adjust if you have a different MPNet variant
embedding_model_strob = SentenceTransformer("NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers")  # Adjust based on the RobBERT model you are using

embeddings_stv1 = np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_sentence_placeholder.npy")
embeddings_stv2 = np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_v2_sentence_placeholder.npy")
embeddings_stmini = np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_miniL12v2_sentence_placeholder.npy")
embeddings_stmp = np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_mpnet_v2_sentence_placeholder.npy")
embeddings_strob = np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_robbert2022_sentence_placeholder.npy")

/workspace/persistent/mijnidbcoachnlp/env-link-3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
import pickle
from gensim.corpora.dictionary import Dictionary

with open("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/tokenized_texts.pkl", "rb") as f:
    tokenized_texts = pickle.load(f)

with open("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/tokenized_texts_stem.pkl", "rb") as f:
    tokenized_texts_stem = pickle.load(f)


dictionary = Dictionary(tokenized_texts)
stem_dictionary = Dictionary(tokenized_texts_stem)

In [7]:
#check tokenized stem words
print(tokenized_texts[4])
print(tokenized_texts_stem[4])

['Beste', 'PERSOON', 'haaruitval', 'kan', 'te', 'maken', 'hebben', 'met', 'medicatiegebruik', 'bij', 'blijvende', 'haarverlies', 'kan', 'ik', 'u', 'naar', 'de', 'dermatoloog', 'sturen', 'voor', 'advies', 'Het', 'bloedverlies', 'ligt', 'niet', 'aan', 'de', 'Purinethol', 'maar', 'heeft', 'te', 'maken', 'met', 'een', 'zieke', 'darm', 'De', 'vraag', 'is', 'of', 'de', 'Purinethol', 'al', 'voldoende', 'werkt', 'Hoeveel', 'Prednison', 'gebruikt', 'u', 'nog', 'Groeten', 'PERSOON']
['best', 'person', 'haaruitval', 'kan', 'te', 'mak', 'hebb', 'met', 'medicatiegebruik', 'bij', 'blijvend', 'haarverlies', 'kan', 'ik', 'u', 'nar', 'de', 'dermatolog', 'stur', 'vor', 'advies', 'het', 'bloedverlies', 'ligt', 'niet', 'aan', 'de', 'purinethol', 'mar', 'heeft', 'te', 'mak', 'met', 'een', 'ziek', 'darm', 'de', 'vrag', 'is', 'of', 'de', 'purinethol', 'al', 'voldoend', 'werkt', 'hoevel', 'prednison', 'gebruikt', 'u', 'nog', 'groet', 'person']


In [8]:
# disable parallelism to avoid some warnings
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 2. Compare ST models with different embeddings

### 2.1 Initialize BERTopic settings fixed configurations

In [9]:
from copy import deepcopy
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# Shared settings (avoid code duplication)
bertopic_settings = {
    "umap_model": UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine'),
    "hdbscan_model": HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=False),
    "vectorizer_model": CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b'),
    "calculate_probabilities": False,
    "verbose": True
}

### 2.2. Define functions for evaluations

In [10]:
from gensim.models.coherencemodel import CoherenceModel
from octis.evaluation_metrics.diversity_metrics import TopicDiversity

def get_top_words(topic_model, top_n):
    """Extract top words for each topic from BERTopic (excluding outliers)."""
    topics = topic_model.get_topics()  # topics is a dict: {topic_num: [(word, score), ...]}

    top_words = []
    for topic_num, word_score_list in topics.items():
        if topic_num == -1:
            continue  # Skip outlier topic (-1)
        words = [word for word, _ in word_score_list[:top_n]]  # Get only the top N words
        top_words.append(words)

    return top_words

    
# Evaluate u_mass coherence
def get_umass(topic_model, tokenized_texts, top_words, dictionary):
    """Calculate C_v coherence score."""
    coherence_model = CoherenceModel(
        topics=top_words,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence='u_mass'
    )
    return coherence_model.get_coherence()

# evaluate nmpi
def get_npmi(topic_model, tokenized_texts, top_words, dictionary):
    """Calculate NPMI coherence using Gensim."""

    coherence_model = CoherenceModel(
        topics=top_words,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence='c_npmi'
    )
    return coherence_model.get_coherence()


def get_outlier_proportion(topic_model):
    """Calculate the percentage of documents labeled as outliers (topic=-1)."""
    topics = topic_model.topics_
    outlier_count = list(topics).count(-1)
    return outlier_count / len(topics)


def get_topic_diversity(top_words, topk=10):
    """
    Compute topic diversity using OCTIS's TopicDiversity metric.
    
    Args:
        top_words: List of lists, where each sublist contains top words for a topic 
                  (e.g., [["word1", "word2"], ["word3", "word4"], ...]).
        topk: Number of top words to consider (must match the length of sublists in top_words).
    
    Returns:
        diversity_score: Float between 0 (low diversity) and 1 (high diversity).
    """
    metric = TopicDiversity(topk=topk)
    diversity_score = metric.score({"topics": top_words})  # OCTIS expects {"topics": ...}
    return diversity_score


In [17]:
def evaluate_model(topic_model, tokenized_texts, tokenized_texts_stem, dictionary, stem_dictionary, embeddings):
    results = {}
    
    # Get Top Words
    top_words = get_top_words(topic_model, 10)

    stem_top_words = stem_tokens(stemmer, top_words)
    
    # Coherence Scores
    if tokenized_texts is not None and dictionary is not None:
        umass_score = get_umass(topic_model, tokenized_texts, top_words, dictionary)
        stem_umass_score = get_umass(topic_model, tokenized_texts_stem, stem_top_words, stem_dictionary)
        npmi_score = get_npmi(topic_model, tokenized_texts, top_words, dictionary)
        results['u_mass_coherence'] = umass_score
        results['u_mass_coherence_stemmed'] = stem_umass_score
        results['npmi_coherence'] = npmi_score
    else:
        results['u_mass_coherence'] = None
        results['npmi_coherence'] = None
    
    # Outlier Proportion
    outlier_prop = get_outlier_proportion(topic_model)
    results['outlier_proportion'] = outlier_prop

    # Topic Diversity
    topic_diversity = get_topic_diversity(top_words)
    results['topic_diversity'] = topic_diversity

    # Top words
    
    print("\n--- Model Evaluation Metrics ---")
    for metric, value in results.items():
        print(f"{metric}: {value:.4f}" if isinstance(value, (int, float)) and value is not None else f"{metric}: {value}")
    for topics in top_words[:4]:
        print(topics)
    print("--------------------------------\n")

    return results


### 2.3 Run models with different embeddings and save the models 

In [12]:

# Define model names and embeddings
models_and_embeddings = [
    ("stv1", "sentence-transformers/distiluse-base-multilingual-cased-v1", embeddings_stv1),
    ("stv2", "sentence-transformers/distiluse-base-multilingual-cased-v2", embeddings_stv2),
    ("mini", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", embeddings_stmini),
    ("mpnet", "sentence-transformers/paraphrase-multilingual-mpnet-base-v2", embeddings_stmp),
    ("robbert", "NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers", embeddings_strob),
]


## 3 Evaluate models with different embeddings

### 3.1 Run each model 3 times

In [ ]:
# DO NOT RUN
# fit all the models once 
from bertopic import BERTopic
import joblib
from gensim.corpora.dictionary import Dictionary

runs = [0, 1, 2]
for run_idx in runs:
    for name, embedding_model, embeddings in models_and_embeddings:
        print(f"Training model: {name} (Run {run_idx + 1})")

        # Train BERTopic
        topic_model = BERTopic(**bertopic_settings)
        topics, probs = topic_model.fit_transform(documents=sentences, embeddings=embeddings)

        # Optional: Save the model
        topic_model.save(f"/workspace/persistent/mijnidbcoachnlp/saved_models/st_models_for_comparison/st_{name}_{run_idx + 1}", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

print("Finished.")



2025-05-01 13:35:58,341 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training model: stv1 (Run 1)


2025-05-01 13:36:17,828 - BERTopic - Dimensionality - Completed ✓
2025-05-01 13:36:17,832 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-01 13:36:31,030 - BERTopic - Cluster - Completed ✓
2025-05-01 13:36:31,047 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-05-01 13:36:32,064 - BERTopic - Representation - Completed ✓
2025-05-01 13:36:33,390 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training model: stv2 (Run 1)


2025-05-01 13:36:49,883 - BERTopic - Dimensionality - Completed ✓
2025-05-01 13:36:49,887 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-01 13:36:51,726 - BERTopic - Cluster - Completed ✓
2025-05-01 13:36:51,740 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-05-01 13:36:52,697 - BERTopic - Representation - Completed ✓
2025-05-01 13:36:54,137 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


...

Training model: mpnet (Run 3)


2025-05-01 13:40:55,705 - BERTopic - Dimensionality - Completed ✓
2025-05-01 13:40:55,709 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-01 13:40:57,606 - BERTopic - Cluster - Completed ✓
2025-05-01 13:40:57,621 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-05-01 13:40:58,685 - BERTopic - Representation - Completed ✓
2025-05-01 13:41:00,301 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training model: robbert (Run 3)


2025-05-01 13:41:17,389 - BERTopic - Dimensionality - Completed ✓
2025-05-01 13:41:17,392 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-01 13:41:19,028 - BERTopic - Cluster - Completed ✓
2025-05-01 13:41:19,043 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-05-01 13:41:20,077 - BERTopic - Representation - Completed ✓


Finished.


### 3.2 Evaluate each run of each model

In [18]:
import os
from bertopic import BERTopic

all_results = {}

run_indices = range(3)  # or use range(N) if you're doing multiple runs

for run_index in run_indices:
    idx = run_index + 1
    for name, embedding_model, embeddings in models_and_embeddings:        
        model_path = f"/workspace/persistent/mijnidbcoachnlp/saved_models/st_models_for_comparison/st_{name}_{idx}"
        print(model_path)
        topic_model = BERTopic.load(model_path, embedding_model=embedding_model)
        
        results = evaluate_model(topic_model, tokenized_texts, tokenized_texts_stem, dictionary, stem_dictionary, embeddings)
        key = f"{name}_run_{idx}"
        # Store results with a unique key
        all_results[key] = results

/workspace/persistent/mijnidbcoachnlp/saved_models/st_models_for_comparison/st_stv1_1


100%|██████████| 321/321 [00:00<00:00, 7587.59it/s]



--- Model Evaluation Metrics ---
u_mass_coherence: -12.3036
u_mass_coherence_stemmed: -10.6134
npmi_coherence: -0.1284
outlier_proportion: 0.4086
topic_diversity: 0.6327
['bloed', 'prikken', 'bloedprikken', 'slijm', 'ontlasting', 'bloeduitslagen', 'laten', 'geprikt', 'ingeleverd', 'inleveren']
['medicatie', 'medicijnen', 'medicijn', 'apotheek', 'recept', 'voorbereiding', 'medicijngebruik', 'ontvangen', 'picoprep', 'narcose']
['vraag', 'vragen', 'vragenlijst', 'vraagje', 'vraagjes', 'daarnaast', 'ingevuld', 'vragenlijsten', 'vandaar', 'vergeten']
['morgen', 'vandaag', 'bereikbaar', 'hele', 'ochtend', 'morgenvroeg', 'ene', 'morgens', 'fijne', 'dagelijks']
--------------------------------

/workspace/persistent/mijnidbcoachnlp/saved_models/st_models_for_comparison/st_stv2_1


100%|██████████| 323/323 [00:00<00:00, 9689.73it/s]


KeyboardInterrupt: 

In [ ]:
import pandas as pd
results_df = pd.DataFrame(all_results)
results_df

0                                          \
                        stv1      stv2      mini     mpnet   robbert   
c_v_coherence       0.350739  0.367847  0.365752  0.376415  0.373420   
npmi_coherence     -0.132329 -0.126118 -0.113201 -0.111450 -0.108439   
silhouette_score    0.042389  0.046476  0.052482  0.064829  0.057004   
outlier_proportion  0.408619  0.406017  0.342299  0.341618  0.303947   
topic_diversity     0.632710  0.596904  0.622928  0.609014  0.645455   

                           1                                          \
                        stv1      stv2      mini     mpnet   robbert   
c_v_coherence       0.354039  0.373360  0.373295  0.373483  0.382440   
npmi_coherence     -0.131716 -0.123427       NaN -0.113467 -0.102586   
silhouette_score    0.039370  0.039280  0.046772  0.066667  0.060862   
outlier_proportion  0.413629  0.386391  0.297478  0.369634  0.305090   
topic_diversity     0.630744  0.605678  0.618563  0.603621  0.633696   

                           2                                          
                        stv1      stv2      mini     mpnet   robbert  
c_v_coherence       0.353667  0.363730  0.365547  0.380882  0.379972  
npmi_coherence     -0.134397 -0.123044 -0.120096       NaN -0.100194  
silhouette_score    0.045798  0.036223  0.052164  0.064857  0.058607  
outlier_proportion  0.408351  0.389723  0.338311  0.370218  0.301102  
topic_diversity     0.641563  0.600000  0.618132  0.606557  0.639276

In [ ]:
import pandas as pd

# Convert list of dicts to a DataFrame
results_df = pd.DataFrame(all_results)

# Save to CSV
results_df.to_csv("/workspace/persistent/mijnidbcoachnlp/data/results/model_evaluation_results_3runs.csv", index=False)

print("Results saved successfully!")

In [32]:
model_path = f"/workspace/persistent/mijnidbcoachnlp/saved_models/st_models_for_comparison/st_mini_3"
topic_model = BERTopic.load(model_path, embedding_model=embedding_model_stmini)
topic_info = topic_model.get_topic_info()
topic_info.head(5)



,Topic,Count,Name,Representation,Representative_Docs
0,-1,13911,-1_afspraak_buikpijn_darmen_buik,"[afspraak, buikpijn, darmen, buik, contact, la...",NaN
1,0,2893,0_bloed_prikken_bloedprikken_bloeduitslagen,"[bloed, prikken, bloedprikken, bloeduitslagen,...",NaN
2,1,881,1_medicatie_medicijnen_zetpillen_medicijn,"[medicatie, medicijnen, zetpillen, medicijn, p...",NaN
3,2,602,2_test_thuistest_testen_zelftest,"[test, thuistest, testen, zelftest, gedaan, th...",NaN
4,3,531,3_toilet_vaker_rennen_toiletbezoek,"[toilet, vaker, rennen, toiletbezoek, toiletbe...",NaN


In [33]:
# Filter out rows with empty strings in "Representation"
filtered_topics = topic_info[
    topic_info["Representation"].apply(lambda x: "" in x)
]
filtered_topics

,Topic,Count,Name,Representation,Representative_Docs
57,56,105,56_hoor_graag__,"[hoor, graag, , , , , , , , ]",NaN
96,95,70,95_voorbaat_dank_plaatsvindt_binnenkort,"[voorbaat, dank, plaatsvindt, binnenkort, zorg...",NaN
167,166,42,166_hoor_graag__,"[hoor, graag, , , , , , , , ]",NaN
185,184,38,184_hoor_liefs_zsm_groetjes,"[hoor, liefs, zsm, groetjes, gehoord, zie, gro...",NaN
196,195,35,195_hoor_graag_groetjes_kim,"[hoor, graag, groetjes, kim, groeten, binnenko...",NaN
212,211,32,211_hoor_graag__,"[hoor, graag, , , , , , , , ]",NaN
240,239,28,239_snelle_reactie_dank_dankjewel,"[snelle, reactie, dank, dankjewel, bedankt, re...",NaN
247,246,27,246_bericht_dank_hartelijk_beste,"[bericht, dank, hartelijk, beste, mededeling, ...",NaN
255,254,26,254_hoor_vrgr_graag_groet,"[hoor, vrgr, graag, groet, tel, mvg, groeten, ...",NaN
261,260,26,260_reactie_alvast_hartelijk_bedankt,"[reactie, alvast, hartelijk, bedankt, dank, no...",NaN


,Topic,Count,Name,Representation,Representative_Docs
58,57,105,57_hoor_graag__,"[hoor, graag, , , , , , , , ]",NaN
96,95,69,95_voorbaat_dank_plaatsvindt_hopelijk,"[voorbaat, dank, plaatsvindt, hopelijk, alvast...",NaN
168,167,42,167_hoor_graag__,"[hoor, graag, , , , , , , , ]",NaN
190,189,35,189_hoor_graag_groetjes_kim,"[hoor, graag, groetjes, kim, groeten, binnenko...",NaN
205,204,32,204_hoor_graag__,"[hoor, graag, , , , , , , , ]",NaN
217,216,30,216_hoor_zsm_zie_beste,"[hoor, zsm, zie, beste, , , , , , ]",NaN
233,232,28,232_snelle_reactie_dank_bedankt,"[snelle, reactie, dank, bedankt, dankjewel, re...",NaN
240,239,27,239_bericht_mededeling_dank_hartelijk,"[bericht, mededeling, dank, hartelijk, beste, ...",NaN
247,246,26,246_slim_vervolg_aanhouden_verstandig,"[slim, vervolg, aanhouden, verstandig, gedaan,...",NaN
250,249,26,249_reactie_alvast_hartelijk_bedankt,"[reactie, alvast, hartelijk, bedankt, dank, no...",NaN
